<a href="https://colab.research.google.com/github/nainikadevireddy/JohnsHopkinsAI/blob/main/Deep%20Neural%20Networks/8%3A%20Radon%20Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>☢️Radon</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

In [ ]:
#!pip install --upgrade --force-reinstall --no-deps kaggle >> log  # upgrade kaggle package (to avoid a warning)
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log       # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 10mar25-radon # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: 10mar25-radon
Using competition: 10mar25-radon
  teamId  teamName                   submissionDate              score          
--------  -------------------------  --------------------------  -------------  
13579215  Team14_Sichao_Shrinit      2025-03-31 00:46:22.476000  29.4472128539  
13505058  Team 11                    2025-03-28 06:31:38.826000  29.6880655424  
13542127  David_Shunguan_Team_8      2025-03-31 00:00:14.890000  34.5457811008  
13548885  Group9_ Nainika_ Arvin     2025-03-31 00:58:16.650000  35.2265109131  
13549098  Group4_Ray_Shailesh        2025-03-31 00:58:33.760000  36.5209640470  
13542791  Jenelle_Caz_Team_1         2025-03-30 00:02:44.426000  39.1130734966  
13515846  Seth Barshay               2025-03-30 18:22:30.190000  40.7723162583  
13559487  Team 13                    2025-03-30 17:59:01.340000  43.4894788248  
13501390  3 Ajide Simmons            2025-03-31 01:19:21.

In [ ]:
# !pip install inflect==7.0.0 >> log  # resolves pip's dependency issue for inflect 7.4.0 requires typeguard>=4.0.1
# !pip install -U tensorflow_addons uszipcode >> log
# !pip install 'keras<3.0.0' mediapipe-model-maker --no-deps >>log # fix from https://github.com/google-ai-edge/mediapipe/issues/5229
# !pip install "pyyaml>6.0.0" "keras<3.0.0" "tensorflow<2.16" "tf-models-official<2.16" mediapipe-model-maker --no-deps >> log  # https://github.com/google-ai-edge/mediapipe/issues/5229

In [ ]:
%%time
%%capture log_capture
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all" # multi-output from a cell
import numpy as np, pandas as pd, time, tensorflow as tf, os #, tensorflow_addons as tfa, tensorflow.keras as keras
from keras.layers import Flatten, Dense
os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'; # allows seeding RNG on GPU
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 3.88 s, sys: 676 ms, total: 4.56 s
Wall time: 4.43 s


In [ ]:
df_raw = pd.read_csv('XY_radon.csv'); df_raw

,Uppm,adjwt,basement,cntyfips,county,dupflag,floor,lat,lon,pcterr,...,stfips,stopdt,stoptm,stratum,typebldg,wave,windoor,zip,zipflag,Y
0,1.80,54.97,Y,59,MORTON,0,1,46.66,-101.39,7.76,...,38,32206,1230,2,2,1,NaN,58554,0,NaN
1,1.65,499.34,N,85,KOSCIUSKO,0,1,40.85,-86.22,55.02,...,18,11497,1430,3,1,32,NaN,46580,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12571,0.44,394.07,Y,3,ANOKA,0,0,44.91,-92.86,9.40,...,27,32110,1500,2,1,4,NaN,55303,0,8.6
12572,2.71,157.82,Y,15,MOHAVE,0,0,36.01,-113.21,14.46,...,4,11496,1330,1,1,38,NaN,86403,0,1.9


In [ ]:
tmr = Timer()

⏳ started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳Timed Green Playground (TGP): Your ideas, code, documentation, and timer START HERE!</font></strong>

<font color=green>Students: Keep all your definitions, code, documentation in <b>TGP</b>. Modifying any code outside of TGP incurs penalties.

<font color=green><h3><b>$\alpha$. Split observations into train and test sets \+ preprocessing</b><h3>


In [ ]:
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

np.random.seed(0)
tf.random.set_seed(0)

# set-up

df = df_raw.copy()
df = df.drop(columns=["windoor", "state2", "zipflag", "zip", "county", "cntyfips"], errors='ignore')

# Feature engineering
df["duration"] = df["stopdt"] - df["startdt"]
df["Uppm_log"] = np.log1p(df["Uppm"])
df["room_log"] = np.log1p(df["room"])
df["Uppm_x_floor"] = df["Uppm"] * df["floor"]
df["floor_is_0"] = (df["floor"] == 0).astype(int)

# train/test split
vX = df.query('Y!=Y').drop('Y', axis=1)
tXY = df.query('Y==Y').copy()
tXY["Y_log"] = np.log1p(tXY["Y"])
tX = tXY.drop(columns=["Y", "Y_log"])
tY_log = tXY["Y_log"]

# Feature lists
num_features = [
    "Uppm_log", "room_log", "duration", "Uppm_x_floor", "floor_is_0",
    "lat", "lon", "pcterr", "adjwt", "rep", "wave"
]
cat_features = ["basement", "floor", "state"]

# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

<font color=green><h3><b>$\beta$. Build and train a model</b><h3>

In [ ]:


# model

def build_model(input_dim):
    tf.random.set_seed(0)
    Init = keras.initializers.RandomNormal(seed=0)
    model = keras.models.Sequential([
        keras.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation="relu", kernel_initializer=Init),
        keras.layers.Dense(64, activation="relu", kernel_initializer=Init),
        keras.layers.Dense(32, activation="relu", kernel_initializer=Init),
        keras.layers.Dense(1, kernel_initializer=Init)
    ])
    model.compile(loss="mse", optimizer="adam", metrics=["mse"])
    return model

# cross-validation

#kf = KFold(n_splits=5, shuffle=True, random_state=42)
#val_scores_log = []
#val_scores_kaggle = []

#for train_idx, val_idx in kf.split(tX):
#    X_train_raw, X_val_raw = tX.iloc[train_idx], tX.iloc[val_idx]
#    y_train_log, y_val_log = tY_log.iloc[train_idx], tY_log.iloc[val_idx]

#    preprocessor_fold = ColumnTransformer([
#        ("num", numeric_transformer, num_features),
#        ("cat", categorical_transformer, cat_features)
#    ])
#    X_train = preprocessor_fold.fit_transform(X_train_raw)
#    X_val = preprocessor_fold.transform(X_val_raw)

#    model = build_model(input_dim=X_train.shape[1])
#    cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

#    model.fit(X_train, y_train_log, validation_data=(X_val, y_val_log),
#              batch_size=32, epochs=100, callbacks=[cb], verbose=0)

#    y_val_pred_log = model.predict(X_val).flatten()
#    y_val_pred = np.expm1(y_val_pred_log)
#    y_val_true = np.expm1(y_val_log)

#    mse_log = mean_squared_error(y_val_log, y_val_pred_log)
#    mse_kaggle = mean_squared_error(y_val_true, y_val_pred)

#    val_scores_log.append(mse_log)
#    val_scores_kaggle.append(mse_kaggle)

# validation scores

#print(f"Log-space MSEs (for reference): {val_scores_log}")
#print(f"Mean log MSE: {np.mean(val_scores_log):.4f}")

#print(f"Kaggle-style MSEs (original Y): {val_scores_kaggle}")
#print(f"Mean Kaggle-style MSE: {np.mean(val_scores_kaggle):.4f}")

# train on full dataset

X_full = preprocessor.fit_transform(tX)
X_test = preprocessor.transform(vX)

final_model = build_model(input_dim=X_full.shape[1])
cb = keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)

final_model.fit(X_full, tY_log, batch_size=64, epochs=90, callbacks=[cb], verbose=1)



Epoch 1/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.5395 - mse: 1.5395
Epoch 2/90
33/99 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3653 - mse: 0.3653

/usr/local/lib/python3.11/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mse
  current = self.get_monitor_value(logs)


99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3638 - mse: 0.3638
Epoch 3/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3445 - mse: 0.3445
Epoch 4/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3289 - mse: 0.3289
Epoch 5/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3139 - mse: 0.3139
Epoch 6/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2968 - mse: 0.2968
Epoch 7/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2745 - mse: 0.2745
Epoch 8/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2487 - mse: 0.2487
Epoch 9/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2226 - mse: 0.2226
Epoch 10/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1976 - mse: 0.1976
Epoch 11/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1745 - mse: 0.1745
Epoch 12/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1541 - mse: 0.1541
Epoch 13/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1386 - mse: 0.1386
Epoch 14/90
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 

<font color=green><h3><b>$\gamma$. Make predictions</b><h3>

In [ ]:
# final prediction

y_final = np.expm1(final_model.predict(X_test).flatten())
pY = pd.DataFrame(y_final, index=np.arange(len(vX)) + 1, columns=['y'])
ToCSV(pY.round(2), 'attempt-final')

197/197 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


<font color=green><h3><b>$\delta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: Feature Engineering
1. **Idea Discovery**: The original features in the dataset consisted of a mixture of categorical and numerical features that, in its raw form, did not capture the relationship between the target variable and the features well. Feature engineering was explored to represent the data in a more informative way
1. **Finding's Importance**: Raw features like Uppm and room had long-tailed distributions, making training unstable. Engineered features reduced skew, improved representation of basement/floor dynamics, and added physical context to the model. Log-transformed uranium concentration and room numbers reduced skew and helped stabilize learning. We introduced a new feature ‘Uppm_x_floor’ which multiplies the uranium concentration with the floor the entry was taken; this was to model how radon entry varies with uranium and elevation. Duration was calculated instead of using the start and stop time-based features. A boolean variable ‘floor_is_0’ is used to capture basement-level measurements where radon can be the highest. Categorical variables were one-hot encoded to represent non-ordinal categories better.
1. **Experiment Setup**: Each feature was added/modified individually into the dataset. All features were standardized using StandardScaler, or one-hot encoded. The dataset with the new feature engineering changes (indivdually added) were inputted into a neural network of depth 3. The mean square error was calculated in a 5-cross validation set-up.
1. **Results**: Overall, the use of the newly engineered features decreased the mean square error from the baseline of 80.0304 to 67.4383.

1. **Title**: Target Transformation
1. **Idea Discovery**: Initial attempts of training with the raw Y values resulted in an unstable loss and very large variance in the predictions. Applying log1p transformation to the target variable was explored to normalize the distribution.
1. **Finding's Importance**: Radon concentration spans several orders of magnitude. Using log1p(Y) stabilized the regression problem, reduced sensitivity to extreme values, and improved gradient behavior during training.

1. **Experiment Setup**:The model was trained on log1p(Y) and predictions were transformed back using expm(1) to match the Kaggle leaderboard scale
1. **Results**: The use of target transformation resulted in significantly increased stability in training. The MSE across 5-cross validation folds reduced from 67.4383 to 43.4239

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title:** Network Architecture Optimization
1. **Idea Discovery:** The baseline model used small layers (5 neurons each) which limited the model's representational capacity. We thought that a deeper and wider architecture would better capture the complex relationships in the data
1. **Finding's Importance:** The right architecture size balances complexity and generalization. Too small networks could underfit, while too large ones risk overfitting and slower training. Our experiments showed that moderately sized layers (64-64-32) provided the best performance.
1. **Experiment Setup:** We systematically tested different architectures, varying both width (neurons per layer) and depth (number of layers), keeping other parameters constant. Each model was evaluated using the log-transformed target.
1. **Results:** The best architecture we tested with 64-64-32 neurons resulted in approximately halving the MSE compared to the original architecture. Adding more neurons (100-100-50) actually increased the MSE from 35.226 to 38.557, suggesting overfitting with larger architectures.

1. **Title**: Hyperparameter Tuning
1. **Idea Discovery**: Neural network performance is sensitive to parameters like batch size, learning rate, and training duration. We tested these hyperparameters to find the optimal configuration
1. **Finding's Importance**: Proper hyperparameter selection affects model convergence and generalization. Batch size influences training stability and speed, while early stopping prevents overfitting by halting training when validation performance deteriorates.
1. **Experiment Setup**: We tested different batch sizes including 32 and 64, and implemented early stopping with patience=8. We also experimented with different training durations (epochs) while monitoring validation loss to prevent overfitting.
1. **Results**: Using a batch size of 64 proved optimal for our model. Reducing the batch size to 32 required removing 10 epochs due to the time constraint, and the MSE increased significantly from 35.2265109131 to 40.3302297804. The early stopping mechanism never kicked in for our model, so we ran as many epochs as the time limit would allow.

1. **Title**: Dropout layers
1. **Idea Discovery**: We looked at adding dropout layers due to their ability to improve generalization
1. **Finding's Importance**: While dropout is a common method to reduce overfitting in deep neural networks, its effectiveness depends on the specific problem and data characteristics.
1. **Experiment Setup**: We tested models with dropout layers added between dense layers, using various dropout rates (0.1, 0.2, 0.3).
1. **Results**: Using dropout layers ended up significantly hurting performance. In all configurations we tested, it would come close to doubling the MSE compared to not using dropout. This showed that this dataset benefits  from preserving all information during training, so we did not include dropout.

<font color=green><h3><b>$\epsilon$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. Analytics Vidhya. "Feature Transformation and Scaling Techniques to Boost Your Model Performance." June 2024. https://www.analyticsvidhya.com/blog/2020/07/types-of-feature-transformation-and-scaling/
1. Kolouri, S., et al. "Neural Networks, Hypersurfaces, and Radon Transforms." arXiv preprint, 2019. https://arxiv.org/abs/1907.02220


<font size=5>⌛</font> <strong><font color=green size=5>Do not exceed competition's runtime limit! Do not write code outside TGP</font></strong>
<hr color=green size=40>

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 47 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>

1. Try different DNN architectures and tuning of hyperparameters
1. Try converting locations to distances to the key Radon sources (which you might need to discover).
1. Try clustering categorical variables by their relation to Radon levels
1. Try replacing categorical values with their level frequencies or other encodings
1. Try scaling features linearly or nonlinearly
1. Try embedding **textual** values (eg. US States names) with pre-trained SBERT-like models. This injects some additional information from Wikipedia (or whichever corpora were used for model training).
1. Do EDA and understand the variables and their relation to the output. [Example 1](https://www.pymc.io/projects/examples/en/latest/generalized_linear_models/multilevel_modeling.html), [Example 2](https://www.tensorflow.org/probability/examples/Multilevel_Modeling_Primer)

<hr>
<font color=black>
    <details><summary><font color=carnelian>▶ </font>Clustering categorical variables <b></b>.</summary>

  1. When we represent categorical variables as dummies, we may be losing important multivariate information. For example, say we use weekdays to predict the number of hours a person works. We could convert weekdays to 6 features (one is dropped due to collinearity). This requires 6 coefficients (degrees of freedom or sources of uncertainty). Essentially, we have an overparameterized model, whereas all we really need is two clusters of categorical values - weekends (Sat/Sun) and non-weekends (M/T/W/Th/F). In general, the model overparameterized model will do worse due to higher variance of the model output (resulting from the overfit and higher flexibility).

  1. Here is another example from the NLP domain, where each word is a feature (or dimension). While morphological variants of a word (eg. run, running, runner, ran, runs, ...) have lower frequency, we cluster them into the same lemma "run", assuming only a small loss of semantic information. We hope that the gain in building a better distribution estimate for "run" is greater than the loss of semantic and lexical information.
        </details>
    <details><summary><font color=carnelian>▶ </font>Distance to Radon source<b></b>.</summary>

If you can determine where Radon is most active (i.e. the source), then you might be able to compute the distance to the source. Ordinarily, we expect lower radiation for greater distance from the source (assuming uniform distribution of underground rivers, geology, rains/winds and other weather conditions affecting distribution of radon, etc.). You could also use categorical features in (e.g. US State, region, etc.), but these might perform better when clustered (again). Distance to the source is a real-valued feature, which does not require clustering.
        </details>
</font>

</div> </details>